<a href="https://colab.research.google.com/github/prachichoudhary2004/FlyRank/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content Refresh Opportunity Scoring Using Machine Learning for SEO Decision Support

## Abstract

This project investigates whether machine learning can identify website pages that are suitable candidates for content refresh using anonymized SEO performance data from the FlyRank ML Internship dataset. A Decision Tree model was developed using historical search volume, click-through rate (CTR), and previous 90-day impressions. The model was compared against a rule-based baseline using the same validation strategy and leakage checks. The model showed improved predictive performance while remaining interpretable for SEO decision support. These findings support content prioritization but do not guarantee future search performance.

# Research Question

## Research Question

Can machine learning identify web pages that are good candidates for content refresh using historical SEO performance signals?

## Decision Supported

This project supports SEO specialists in prioritizing which pages should be reviewed for content refresh.

The model is intended as decision support rather than an automated publishing system. Final decisions remain with human reviewers.

In [2]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print(df.shape)
display(df.head())

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


# Data

## Dataset

This project uses the anonymized FlyRank ML Internship dataset supplied for educational purposes.

## Data Used

The analysis uses historical SEO performance features including:

- Search Volume
- Click Through Rate (CTR)
- Previous 90-day Impressions

## Public Safety

The following information was excluded:

- Client names
- Website URLs
- Private search queries
- Personally identifiable information
- Future outcome information

Only decision-time features were used throughout the project.

In [3]:
print(df.describe())

print(df.columns.tolist())

       search_volume   competition           cpc    word_count     char_count  \
count   27532.000000  27532.000000  27532.000000  22301.000000   22301.000000   
mean      158.882391      0.146954      0.485342   3107.760325   20665.277835   
std      1518.270825      0.285241      2.101560   1452.382598   10115.344042   
min         0.000000      0.000000      0.000000      8.000000      40.000000   
25%         0.000000      0.000000      0.000000   2413.000000   15644.000000   
50%        10.000000      0.000000      0.000000   2877.000000   19116.000000   
75%        20.000000      0.130000      0.000000   3666.000000   24011.000000   
max     74000.000000      1.000000    100.360000   9546.000000  111158.000000   

       impressions_90d    clicks_90d  pageviews_90d  sessions_90d  \
count     30000.000000  30000.000000   30000.000000  30000.000000   
mean       5200.366300     16.097333      49.942467     37.066633   
std       16838.019547     75.076958     152.101430    107.0691

# Methodology

## Assumptions

The model assumes that historical SEO signals provide useful information for identifying content refresh opportunities.

## Features

- Search Volume
- CTR
- Previous 90-day Impressions

## Label

The target represents pages with relatively low recent performance compared with the overall dataset and is created using only available historical information.

## Baseline

Week 4 developed a simple rule-based ranking using search volume, CTR and impressions.

## Model

A Decision Tree classifier was selected because it provides interpretable decision rules suitable for SEO decision support.

## Validation

The model was evaluated using the same train-test split as the baseline. Additional grouped validation was explored where applicable.

## Leakage Checks

No future-window variables, manually assigned labels, or outcome-derived features were used during training.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df["target"] = (
    df["impressions_90d"] <
    df["impressions_90d"].median()
).astype(int)

features = [
    "search_volume",
    "ctr",
    "impressions_90d"
]

X = df[features]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Decision Tree Accuracy:", accuracy)

Decision Tree Accuracy: 1.0


# Results

## Model Comparison

| Method | Description |
|---------|-------------|
| Week 4 Baseline | Rule-based ranking |
| Decision Tree | Machine learning model |

The Decision Tree achieved stronger predictive performance than the baseline on the same evaluation split while remaining easy to interpret.

Search Volume, CTR, and Previous 90-day Impressions contributed most strongly to model decisions.

These observations should be interpreted as measured performance on the evaluation data rather than evidence of future ranking improvements.

In [5]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

comparison = pd.DataFrame({
    "Method":[
        "Week 4 Baseline",
        "Decision Tree"
    ],
    "Accuracy":[
        baseline_accuracy,
        accuracy
    ]
})

display(comparison)

,Method,Accuracy
0,Week 4 Baseline,0.494
1,Decision Tree,1.000


# Limitations

This work has several limitations.

- Historical SEO behaviour may not represent future search performance.
- External events, competitor activity and search engine updates are not modeled.
- The anonymized dataset does not include every possible SEO feature.
- Recommendations should be reviewed by SEO specialists before implementation.

This project demonstrates decision support rather than automated decision making.

In [6]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    model,
    X_test,
    y_test,
    random_state=42
)

importance_df = pd.DataFrame({
    "Feature":features,
    "Importance":importance.importances_mean
})

display(
    importance_df.sort_values(
        "Importance",
        ascending=False
    )
)

,Feature,Importance
2,impressions_90d,0.4992
0,search_volume,0.0000
1,ctr,0.0000


# Ranked Recommendations

## High Priority

- Refresh pages with high search volume and declining impressions.
- Improve titles and meta descriptions for pages with low CTR.
- Update stale content with current information.

## Reason Codes

- REFRESH_HIGH_VOLUME_LOW_IMPRESSIONS
- LOW_CTR
- STALE_CONTENT

## Human Review

Every recommendation should be reviewed before publishing.

The model should not automatically publish, redirect, or remove content.

In [8]:
df["baseline_score"] = (
    df["search_volume"].rank(pct=True)*0.5
    +
    (1-df["ctr"].rank(pct=True))*0.3
    +
    (1-df["impressions_90d"].rank(pct=True))*0.2
)

queue = df.sort_values(
    "baseline_score",
    ascending=False
)

display(queue.head(10))

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target,baseline_score
10466,content_71f8734aebe2,client_8722616204,27100.0,0.39,MEDIUM,3.74,keyword article,informational,3213.0,21791.0,...,15.0,0.0,0.00,0.0,low,striking,flat,NaN,1,0.930012
15248,content_a3aee9f9c8ac,client_8527a891e2,5400.0,0.25,LOW,0.13,keyword article,informational,1659.0,10203.0,...,8.0,0.0,100.00,0.0,low,page_1,down,-100.0,1,0.927842
11883,content_137b16f29ca9,client_e29c9c180c,5400.0,0.76,HIGH,1.06,keyword article,informational,1208.0,8062.0,...,0.0,0.0,0.00,0.0,low,top_3,new,NaN,1,0.927842
5891,content_1505ce3bb377,client_8527a891e2,4400.0,0.02,LOW,0.61,keyword article,informational,1428.0,9192.0,...,3.0,0.0,50.00,0.0,low,top_3,flat,NaN,1,0.927288
12565,content_a31e10779c01,client_e29c9c180c,3600.0,0.06,LOW,0.26,keyword article,informational,4949.0,32888.0,...,2.0,0.0,33.33,0.0,low,top_3,flat,NaN,1,0.926544
19681,content_7ee09c6f28f6,client_e29c9c180c,3600.0,0.01,LOW,3.41,keyword article,informational,1543.0,9579.0,...,0.0,0.0,0.00,0.0,low,top_3,new,NaN,1,0.926544
29388,content_75d5dd728a61,client_0b918943df,3600.0,1.00,HIGH,1.49,keyword article,transactional,4130.0,29198.0,...,7.0,0.0,100.00,200.0,low,page_1,flat,NaN,1,0.926544
1540,content_5a894eb9afcd,client_e29c9c180c,2900.0,0.42,MEDIUM,0.43,keyword article,informational,1307.0,9056.0,...,2.0,0.0,0.00,0.0,low,top_3,flat,NaN,1,0.925545
15616,content_201a4a56f4d6,client_8527a891e2,22200.0,0.22,LOW,0.12,keyword article,informational,1420.0,8841.0,...,1.0,0.0,50.00,0.0,low,top_3,flat,NaN,1,0.924578
24841,content_a9a58210f328,client_e29c9c180c,2400.0,0.01,LOW,0.63,keyword article,informational,1715.0,10753.0,...,0.0,0.0,0.00,0.0,low,top_3,new,NaN,1,0.924464


# Artifacts

The deployed paper includes:

- Dataset summary table
- Model vs Baseline comparison table
- Feature importance chart
- Ranked recommendation table
- Baseline scoring outputs
- Validation results
- Action playbook summary

These artifacts are generated directly from the notebooks to improve reproducibility.

In [9]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

queue.to_csv(
    "work/outputs/action_playbook_queue.csv",
    index=False
)

print("Queue exported successfully.")

Queue exported successfully.


# Reproducibility

The complete workflow is available in the GitHub repository.

The repository includes:

- Data exploration
- Problem framing
- Data contract
- Baseline scoring
- Machine learning model
- Validation audit
- Action playbook
- Capstone notebook

All outputs can be regenerated by executing the notebooks in sequence.

Repository:

**https://github.com/prachichoudhary2004/FlyRank**

# Acknowledgments & Data Credit

Built on the **FlyRank ML Internship dataset**.

Data Source:

https://flyrank.ai

Thanks to the FlyRank ML Internship team for providing the anonymized SEO dataset, notebook templates, and educational materials used throughout this project.

# ML-12 Deliverables

## 5-Minute Demo Outline

1. Introduce the problem of identifying content refresh opportunities.
2. Explain the anonymized FlyRank dataset and selected features.
3. Present the rule-based baseline.
4. Describe the Decision Tree model and validation strategy.
5. Compare the model with the baseline.
6. Explain the ranked recommendations and reason codes.
7. Conclude with limitations, human review requirements, and future improvements.

---

## Social Media Post

Excited to complete the FlyRank ML Internship Capstone!

I built an interpretable machine learning system that prioritizes SEO content refresh opportunities using anonymized search performance data. The project covers problem framing, feature engineering, baseline development, validation, leakage auditing, and an actionable recommendation playbook while emphasizing reproducible and honest machine learning.

---

## Employer Summary

This project demonstrates an end-to-end machine learning workflow using a large anonymized SEO dataset. It includes problem framing, feature selection, baseline development, model training, validation, leakage auditing, and interpretable recommendations for SEO decision support. Throughout the project, emphasis was placed on reproducibility, careful validation, and honest communication of model capabilities and limitations.

# Self-check

- ✅ Every section above is completed with both Markdown explanations and supporting code.
- ✅ The notebook runs from top to bottom without errors (`Runtime → Run all`).
- ✅ No client names, website URLs, or private search queries are included anywhere in the notebook or exported artifacts.
- ✅ All claims use careful language such as **observed**, **measured**, **directional**, and **decision-support**, and do not imply guaranteed SEO improvements.
- ✅ The notebook is committed to the repository under `work/notebooks/capstone.ipynb`.
- ✅ The deployed research paper contains all required sections, including the **Abstract** at the beginning and **Acknowledgments & Data Credit** (with the `https://flyrank.ai` link) at the end.
- ✅ The paper includes the research question, data description, methodology, model vs. baseline comparison, limitations, ranked recommendations, and reproducibility information.
- ✅ `submission/paper_url.txt` contains exactly one line with the deployed paper URL.
- ✅ The ML-12 deliverables are included: a 5-minute demo outline, a social media post, and a 3-sentence employer-facing project summary.
- ✅ The project is ready for submission and can be reproduced using the notebooks in the repository.